In [0]:
catalog = "hackathon"
schema = "energy_data"
spark.sql(f"USE {catalog}.{schema}")
print("✅ Connected to hackathon.energy_data")

In [0]:
top_flarers = spark.sql("""
    SELECT 
        o.operator_name,
        ROUND(SUM(e.flare_volume), 2) as total_flare_e3m3,
        ROUND(SUM(e.vent_volume), 2) as total_vent_e3m3,
        ROUND(SUM(e.flare_volume) + SUM(e.vent_volume), 2) as total_emissions,
        COUNT(DISTINCT e.facility_id) as facility_count
    FROM facility_emissions e
    JOIN facilities f ON e.facility_id = f.facility_id
    JOIN operators o ON f.operator_baid = o.operator_baid
    GROUP BY o.operator_name
    ORDER BY total_emissions DESC
    LIMIT 20
""")
top_flarers.display()

In [0]:
flare_intensity = spark.sql("""
    SELECT 
        o.operator_name,
        ROUND(SUM(e.flare_volume) / NULLIF(SUM(v.Volume), 0) * 1000, 4) as flare_intensity_score,
        ROUND(SUM(e.flare_volume), 2) as total_flare,
        ROUND(SUM(v.Volume) * 6.2898, 0) as total_oil_bbl,
        COUNT(DISTINCT e.facility_id) as facilities
    FROM facility_emissions e
    JOIN facilities f ON e.facility_id = f.facility_id
    JOIN operators o ON f.operator_baid = o.operator_baid
    JOIN volumetrics v ON f.facility_id = v.ReportingFacilityID
    WHERE v.ActivityID = 'PROD'
    AND (v.ProductID LIKE '%OIL%' OR v.ProductID LIKE '%CRD%')
    GROUP BY o.operator_name
    HAVING total_oil_bbl > 10000
    ORDER BY flare_intensity_score DESC
    LIMIT 20
""")
flare_intensity.display()

In [0]:
risk_scores = spark.sql("""
    SELECT 
        f.facility_name,
        f.operator_name,
        f.region,
        f.facility_type,
        ROUND(SUM(e.flare_volume), 2) as total_flare,
        ROUND(SUM(e.vent_volume), 2) as total_vent,
        ROUND(SUM(e.flare_volume) + SUM(e.vent_volume), 2) as total_emissions,
        CASE 
            WHEN SUM(e.flare_volume) + SUM(e.vent_volume) > 1000 THEN '🔴 HIGH RISK'
            WHEN SUM(e.flare_volume) + SUM(e.vent_volume) > 500  THEN '🟡 MEDIUM RISK'
            ELSE '🟢 LOW RISK'
        END as regulatory_risk
    FROM facility_emissions e
    JOIN facilities f ON e.facility_id = f.facility_id
    GROUP BY f.facility_name, f.operator_name, f.region, f.facility_type
    ORDER BY total_emissions DESC
    LIMIT 50
""")
risk_scores.display()

In [0]:
flare_intensity.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_operator_scores")
risk_scores.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_risk_scores")
print("✅ FlareWatch tables saved — ready for dashboard!")

In [0]:
price_emissions = spark.sql("""
    SELECT 
        mp.price_month,
        mp.wti_usd_bbl as wti_price,
        mp.wcs_usd_bbl as wcs_price,
        ROUND(SUM(e.flare_volume), 2) as total_flare,
        ROUND(SUM(e.vent_volume), 2) as total_vent,
        ROUND(SUM(e.flare_volume) + SUM(e.vent_volume), 2) as total_emissions
    FROM facility_emissions e
    JOIN facilities f ON e.facility_id = f.facility_id
    JOIN market_prices mp ON DATE_FORMAT(e.production_month, 'yyyy-MM') = DATE_FORMAT(CAST(mp.price_month AS DATE), 'yyyy-MM')
    GROUP BY mp.price_month, mp.wti_usd_bbl, mp.wcs_usd_bbl
    ORDER BY mp.price_month
""")
price_emissions.display()

In [0]:
spark.sql("DESCRIBE hackathon.energy_data.market_prices").display()

In [0]:
price_emissions.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_price_correlation")
print("✅ Price correlation table saved!")

In [0]:
regional_emissions = spark.sql("""
    SELECT 
        f.region,
        f.facility_type,
        COUNT(DISTINCT e.facility_id) as facility_count,
        ROUND(SUM(e.flare_volume), 2) as total_flare,
        ROUND(SUM(e.vent_volume), 2) as total_vent,
        ROUND(SUM(e.flare_volume) + SUM(e.vent_volume), 2) as total_emissions,
        ROUND(AVG(e.flare_volume), 2) as avg_flare_per_facility
    FROM facility_emissions e
    JOIN facilities f ON e.facility_id = f.facility_id
    WHERE f.region IS NOT NULL
    GROUP BY f.region, f.facility_type
    ORDER BY total_emissions DESC
    LIMIT 20
""")
regional_emissions.display()

In [0]:
regional_emissions.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_regional_emissions")
print("✅ Regional emissions table saved!")

In [0]:
regional_summary = spark.sql("""
    SELECT 
        region,
        COUNT(DISTINCT facility_id) as total_facilities,
        COUNT(DISTINCT operator_name) as total_operators,
        ROUND(SUM(flare_volume), 2) as total_flare,
        ROUND(SUM(vent_volume), 2) as total_vent,
        ROUND(AVG(emissions_intensity), 4) as avg_emissions_intensity,
        ROUND(MAX(emissions_intensity), 4) as worst_facility_intensity
    FROM hackathon.energy_data.emissions_intensity
    GROUP BY region
    ORDER BY total_flare DESC
""")
regional_summary.display()

In [0]:
regional_summary.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_regional_summary")
print("✅ Regional summary saved!")

In [0]:
from pyspark.sql.functions import col, when

ml_data = spark.sql("""
    SELECT 
        e.facility_id,
        o.operator_name,
        f.region,
        AVG(e.flare_volume) as avg_flare,
        AVG(e.vent_volume) as avg_vent,
        AVG(ei.emissions_intensity) as avg_intensity,
        MAX(ei.emissions_intensity) as max_intensity,
        COUNT(e.production_month) as months_reported,
        SUM(e.flare_volume) as total_flare,
        SUM(e.vent_volume) as total_vent
    FROM facility_emissions e
    JOIN facilities f ON e.facility_id = f.facility_id
    JOIN operators o ON f.operator_baid = o.operator_baid
    JOIN emissions_intensity ei ON e.facility_id = ei.facility_id
    GROUP BY e.facility_id, o.operator_name, f.region
""")

# Create risk label based on thresholds
ml_data_labeled = ml_data.withColumn("risk_label",
    when(col("avg_intensity") > 0.5, 2)       # HIGH
    .when(col("avg_intensity") > 0.1, 1)       # MEDIUM
    .otherwise(0)                               # LOW
)

ml_data_labeled.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_ml_training")
print(f"✅ Training data saved: {ml_data_labeled.count()} facilities")
ml_data_labeled.display()bbb

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Load training data
df = spark.table("hackathon.energy_data.flarewatch_ml_training").toPandas()

features = ["avg_flare", "avg_vent", "avg_intensity", "max_intensity", 
            "months_reported", "total_flare", "total_vent"]
X = df[features].fillna(0)
y = df["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="FlareWatch_RiskScorer_v1"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(model, "flarewatch_risk_model")
    
    print(f"✅ Model accuracy: {accuracy:.2%}")
    print(classification_report(y_test, y_pred, target_names=["LOW", "MEDIUM", "HIGH"]))

In [0]:
# Updated features - remove the columns we used to create the label
features = ["avg_flare", "avg_vent", "months_reported", "total_flare", "total_vent"]

X = df[features].fillna(0)
y = df["risk_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="FlareWatch_RiskScorer_v2"):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(model, "flarewatch_risk_model")
    
    print(f"✅ Model accuracy: {accuracy:.2%}")
    print(classification_report(y_test, y_pred, target_names=["LOW", "MEDIUM", "HIGH"]))

In [0]:
all_facilities = spark.table("hackathon.energy_data.flarewatch_ml_training").toPandas()
X_all = all_facilities[features].fillna(0)

all_facilities["ml_risk_score"] = model.predict(X_all)
all_facilities["ml_risk_label"] = all_facilities["ml_risk_score"].map({
    0: "🟢 LOW", 1: "🟡 MEDIUM", 2: "🔴 HIGH"
})
all_facilities["ml_risk_confidence"] = model.predict_proba(X_all).max(axis=1).round(3)

result = spark.createDataFrame(all_facilities[["facility_id", "operator_name", "region", 
                                                "ml_risk_label", 
                                                "ml_risk_confidence"]])
result.write.mode("overwrite").saveAsTable("hackathon.energy_data.flarewatch_ml_predictions")
print("✅ ML predictions saved!")
result.orderBy("ml_risk_confidence", ascending=False).display()

In [0]:
from databricks import automl

summary = automl.classify(
    dataset=spark.table("hackathon.energy_data.flarewatch_ml_training"),
    target_col="risk_label",
    primary_metric="accuracy",
    timeout_minutes=10
)

print(f"✅ Best model accuracy: {summary.best_trial.metrics['val_accuracy']:.2%}")
print(f"Best model: {summary.best_trial.model_path}")